# conv-windowing-1d — faded example 1: Window strides for stride-3 1-D conv

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-windowing-1d`. The last cell reports your progress on the `CNN: 1-D conv windowing` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 1-D conv windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-windowing-1d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-windowing-1d"
DD_SUBTOPIC = "CNN: 1-D conv windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For a stride-`S` 1-D convolution the window view is `(B, IC, OW, KW)` with `OW = (W - KW) // S + 1`. The kernel-width axis advances one input element per tap (stride `s_w`), but the output-width axis jumps `S` input elements between windows (stride `S * s_w`). Getting that `OW`-axis stride right is the whole point.

## Faded exercise 1

Complete `conv1d_windows_strided(x, KW, S)` so it returns the `(B, IC, OW, KW)` strided window view for a **stride-`S`** convolution. The shape and the surrounding `as_strided` call are given; you must supply the correct `stride=` tuple. The test contracts your view with a kernel via einsum and compares to `F.conv1d(x, weight, stride=S)`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn.functional as F
from einops import einsum

def conv1d_windows_strided(x: t.Tensor, KW: int, S: int) -> t.Tensor:
    B, IC, W = x.shape
    OW = (W - KW) // S + 1
    s_b, s_ic, s_w = x.stride()
    new_stride = None  # TODO: fill in this step — read the prompt cell above
    return x.as_strided(size=(B, IC, OW, KW), stride=new_stride)


def _test():
    t.manual_seed(0)
    x = t.randn(2, 3, 13)
    weight = t.randn(4, 3, 3)
    KW, S = 3, 3
    win = conv1d_windows_strided(x, KW, S)
    OW = (x.shape[-1] - KW) // S + 1
    assert tuple(win.shape) == (2, 3, OW, KW), win.shape
    assert win.data_ptr() == x.data_ptr(), 'must be a view, not a copy'
    out = einsum(win, weight, 'b i o k, c i k -> b c o')
    ref = F.conv1d(x, weight, stride=S)
    assert t.allclose(out, ref, atol=1e-4), (out - ref).abs().max().item()


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn.functional as F
from einops import einsum

def conv1d_windows_strided(x: t.Tensor, KW: int, S: int) -> t.Tensor:
    B, IC, W = x.shape
    OW = (W - KW) // S + 1
    s_b, s_ic, s_w = x.stride()
    new_stride = (s_b, s_ic, S * s_w, s_w)
    return x.as_strided(size=(B, IC, OW, KW), stride=new_stride)
```
</details>